# Scikit-learn Multiclass Evaluation

Comprehensive evaluation of multiclass predictions against ground truth annotations.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    top_k_accuracy_score,
    log_loss
)
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# CONFIGURE BASE DIRECTORY - Change this to your project directory
BASE_DIR = Path.cwd()  # Current directory
# BASE_DIR = Path('/Users/tod/Desktop/LMM_POC')  # Or set specific path

print(f"📁 Base directory: {BASE_DIR}")

In [ ]:
# Option 1: Load from CSV file
# Uncomment and modify the path below to load your data
# INPUT_FILE = DATA_DIR / 'your_predictions.csv'
# df = pd.read_csv(INPUT_FILE)
# print(f"✅ Loaded data from: {INPUT_FILE}")

# Option 2: Load from specific path
# df = pd.read_csv(Path('/path/to/your/data.csv'))

# Option 3: Use existing DataFrame from previous cell
# df = your_existing_dataframe

# Option 4: Create sample data for testing
np.random.seed(42)
n_samples = 1000
classes = ['class_A', 'class_B', 'class_C', 'class_D', 'class_E']

# Generate ground truth
annotator = np.random.choice(classes, n_samples, p=[0.3, 0.25, 0.2, 0.15, 0.1])

# Generate predictions with some errors
pred = annotator.copy()
error_mask = np.random.random(n_samples) < 0.2  # 20% error rate
pred[error_mask] = np.random.choice(classes, error_mask.sum())

# Create DataFrame
df = pd.DataFrame({
    PRED_COLUMN: pred,
    TRUTH_COLUMN: annotator
})

INPUT_FILE = OUTPUT_DIR / 'sample_data.csv'
df.to_csv(INPUT_FILE, index=False)
print(f"💾 Sample data saved to: {INPUT_FILE}")

print(f"\nDataFrame shape: {df.shape}")
print(f"Unique classes in ground truth: {df[TRUTH_COLUMN].nunique()}")
print(f"Unique classes in predictions: {df[PRED_COLUMN].nunique()}")
df.head(10)

In [ ]:
# Option 1: Load from CSV file
# df = pd.read_csv(BASE_DIR / 'your_predictions.csv')

# Option 2: Use existing DataFrame
# df = your_existing_dataframe

# Option 3: Create sample data for testing
np.random.seed(42)
n_samples = 1000
classes = ['class_A', 'class_B', 'class_C', 'class_D', 'class_E']

# Generate ground truth
annotator = np.random.choice(classes, n_samples, p=[0.3, 0.25, 0.2, 0.15, 0.1])

# Generate predictions with some errors
pred = annotator.copy()
error_mask = np.random.random(n_samples) < 0.2  # 20% error rate
pred[error_mask] = np.random.choice(classes, error_mask.sum())

# Create DataFrame
df = pd.DataFrame({
    'pred': pred,
    'annotator': annotator
})

print(f"DataFrame shape: {df.shape}")
print(f"Unique classes in ground truth: {df['annotator'].nunique()}")
print(f"Unique classes in predictions: {df['pred'].nunique()}")
df.head(10)

## Calculate All Metrics

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground truth distribution
df['annotator'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Ground Truth Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Prediction distribution
df['pred'].value_counts().plot(kind='bar', ax=axes[1], color='lightcoral')
axes[1].set_title('Prediction Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Classification Report

In [ ]:
# Get predictions and ground truth
y_true = df['annotator'].to_numpy()
y_pred = df['pred'].to_numpy()

# Identify valid classes (only those in ground truth)
valid_classes = np.unique(y_true)

# Check for invalid predictions
invalid_mask = ~np.isin(y_pred, valid_classes)
n_invalid = invalid_mask.sum()

if n_invalid > 0:
    invalid_classes = set(np.unique(y_pred)) - set(valid_classes)
    print(f"⚠️ WARNING: Found {n_invalid} predictions ({n_invalid/len(y_pred)*100:.2f}%) with invalid classes")
    print(f"   Invalid classes being predicted: {invalid_classes}")
    for cls in invalid_classes:
        count = (y_pred == cls).sum()
        print(f"     '{cls}': {count} occurrences")
    print("   These will be counted as errors but excluded from visualizations\n")

# Calculate all metrics (including invalid predictions as errors)
metrics = {}

# Basic accuracy (invalid predictions count as errors)
metrics['accuracy'] = accuracy_score(y_true, y_pred)
metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)

# Agreement metrics
metrics['cohen_kappa'] = cohen_kappa_score(y_true, y_pred)
metrics['matthews_corrcoef'] = matthews_corrcoef(y_true, y_pred)

# Per-class metrics - only for valid classes
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=valid_classes, average=None, zero_division=0
)

# Macro averages (only on valid classes)
metrics['precision_macro'] = precision_recall_fscore_support(
    y_true, y_pred, labels=valid_classes, average='macro', zero_division=0)[0]
metrics['recall_macro'] = precision_recall_fscore_support(
    y_true, y_pred, labels=valid_classes, average='macro', zero_division=0)[1]
metrics['f1_macro'] = precision_recall_fscore_support(
    y_true, y_pred, labels=valid_classes, average='macro', zero_division=0)[2]

# Weighted averages
metrics['precision_weighted'] = precision_recall_fscore_support(
    y_true, y_pred, labels=valid_classes, average='weighted', zero_division=0)[0]
metrics['recall_weighted'] = precision_recall_fscore_support(
    y_true, y_pred, labels=valid_classes, average='weighted', zero_division=0)[1]
metrics['f1_weighted'] = precision_recall_fscore_support(
    y_true, y_pred, labels=valid_classes, average='weighted', zero_division=0)[2]

# Store valid classes for later use
metrics['valid_classes'] = valid_classes
metrics['n_invalid_predictions'] = n_invalid

# Display results
print("="*60)
print("OVERALL METRICS")
print("="*60)
print(f"Accuracy:                {metrics['accuracy']:.4f}")
print(f"Balanced Accuracy:       {metrics['balanced_accuracy']:.4f}")
print(f"Cohen's Kappa:           {metrics['cohen_kappa']:.4f}")
print(f"Matthews Corr Coef:      {metrics['matthews_corrcoef']:.4f}")
if n_invalid > 0:
    print(f"Invalid Predictions:     {n_invalid} ({n_invalid/len(y_pred)*100:.2f}%)")

print("\n" + "="*60)
print("AVERAGED METRICS (Valid Classes Only)")
print("="*60)
print("\nMacro Averages (unweighted):")
print(f"  Precision: {metrics['precision_macro']:.4f}")
print(f"  Recall:    {metrics['recall_macro']:.4f}")
print(f"  F1-Score:  {metrics['f1_macro']:.4f}")

print("\nWeighted Averages (by support):")
print(f"  Precision: {metrics['precision_weighted']:.4f}")
print(f"  Recall:    {metrics['recall_weighted']:.4f}")
print(f"  F1-Score:  {metrics['f1_weighted']:.4f}")

## Confusion Matrix

In [ ]:
# Create per-class metrics DataFrame (only for valid classes)
class_metrics = []

for i, cls in enumerate(valid_classes):
    cls_mask = y_true == cls
    cls_pred = y_pred[cls_mask]
    
    # Calculate accuracy including invalid predictions as errors
    accuracy = (cls_pred == cls).mean()
    
    class_metrics.append({
        'class': cls,
        'support': cls_mask.sum(),
        'accuracy': accuracy,
        'precision': precision[i],
        'recall': recall[i],
        'f1': f1[i]
    })

class_df = pd.DataFrame(class_metrics)
class_df = class_df.sort_values('f1', ascending=False)

# Display summary statistics
print("="*70)
print("CLASS PERFORMANCE SUMMARY")
print("="*70)
print(f"Total number of valid classes: {len(class_df)}")
print(f"\nF1-Score Statistics:")
print(class_df['f1'].describe().to_string())

if metrics['n_invalid_predictions'] > 0:
    print(f"\n⚠️ Note: {metrics['n_invalid_predictions']} predictions with invalid classes")
    print("         are counted as errors but not shown as separate classes")

# Show top and bottom performers
print("\n" + "="*70)
print("TOP 10 BEST PERFORMING CLASSES")
print("="*70)
top_10 = class_df.nlargest(10, 'f1')
print(top_10[['class', 'f1', 'precision', 'recall', 'support']].to_string(index=False))

print("\n" + "="*70)
print("TOP 10 WORST PERFORMING CLASSES")
print("="*70)
bottom_10 = class_df.nsmallest(10, 'f1')
print(bottom_10[['class', 'f1', 'precision', 'recall', 'support']].to_string(index=False))

# Create visualizations for many classes
if len(class_df) > 30:
    print(f"\n📊 Detected {len(class_df)} classes - using optimized visualizations for clarity")
    
    # Figure 1: Distribution of metrics
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Precision distribution
    axes[0].hist(class_df['precision'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[0].axvline(class_df['precision'].mean(), color='red', linestyle='--', label=f'Mean: {class_df["precision"].mean():.3f}')
    axes[0].set_xlabel('Precision')
    axes[0].set_ylabel('Number of Classes')
    axes[0].set_title('Distribution of Precision Across Classes')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Recall distribution
    axes[1].hist(class_df['recall'], bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
    axes[1].axvline(class_df['recall'].mean(), color='red', linestyle='--', label=f'Mean: {class_df["recall"].mean():.3f}')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Number of Classes')
    axes[1].set_title('Distribution of Recall Across Classes')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    # F1-Score distribution
    axes[2].hist(class_df['f1'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
    axes[2].axvline(class_df['f1'].mean(), color='red', linestyle='--', label=f'Mean: {class_df["f1"].mean():.3f}')
    axes[2].set_xlabel('F1-Score')
    axes[2].set_ylabel('Number of Classes')
    axes[2].set_title('Distribution of F1-Score Across Classes')
    axes[2].legend()
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.suptitle('Performance Metrics Distribution', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Figure 2: Top 20 performers
    fig, axes = plt.subplots(1, 3, figsize=(15, 8))
    top_20 = class_df.nlargest(20, 'f1')
    
    # Horizontal bars for top performers
    axes[0].barh(range(len(top_20)), top_20['precision'], color='skyblue', alpha=0.8)
    axes[0].set_yticks(range(len(top_20)))
    axes[0].set_yticklabels(top_20['class'], fontsize=8)
    axes[0].set_xlabel('Precision')
    axes[0].set_title('Top 20 Classes - Precision', fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)
    axes[0].invert_yaxis()
    
    axes[1].barh(range(len(top_20)), top_20['recall'], color='lightcoral', alpha=0.8)
    axes[1].set_yticks(range(len(top_20)))
    axes[1].set_yticklabels(top_20['class'], fontsize=8)
    axes[1].set_xlabel('Recall')
    axes[1].set_title('Top 20 Classes - Recall', fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)
    axes[1].invert_yaxis()
    
    axes[2].barh(range(len(top_20)), top_20['f1'], color='lightgreen', alpha=0.8)
    axes[2].set_yticks(range(len(top_20)))
    axes[2].set_yticklabels(top_20['class'], fontsize=8)
    axes[2].set_xlabel('F1-Score')
    axes[2].set_title('Top 20 Classes - F1-Score', fontweight='bold')
    axes[2].grid(axis='x', alpha=0.3)
    axes[2].invert_yaxis()
    
    plt.suptitle('Top 20 Best Performing Classes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Figure 3: Bottom 20 performers (if there are enough classes)
    if len(class_df) > 20:
        fig, axes = plt.subplots(1, 3, figsize=(15, 8))
        bottom_20 = class_df.nsmallest(20, 'f1')
        
        axes[0].barh(range(len(bottom_20)), bottom_20['precision'], color='#FF6B6B', alpha=0.8)
        axes[0].set_yticks(range(len(bottom_20)))
        axes[0].set_yticklabels(bottom_20['class'], fontsize=8)
        axes[0].set_xlabel('Precision')
        axes[0].set_title('Bottom 20 Classes - Precision', fontweight='bold')
        axes[0].grid(axis='x', alpha=0.3)
        axes[0].invert_yaxis()
        
        axes[1].barh(range(len(bottom_20)), bottom_20['recall'], color='#FF9999', alpha=0.8)
        axes[1].set_yticks(range(len(bottom_20)))
        axes[1].set_yticklabels(bottom_20['class'], fontsize=8)
        axes[1].set_xlabel('Recall')
        axes[1].set_title('Bottom 20 Classes - Recall', fontweight='bold')
        axes[1].grid(axis='x', alpha=0.3)
        axes[1].invert_yaxis()
        
        axes[2].barh(range(len(bottom_20)), bottom_20['f1'], color='#FFB3B3', alpha=0.8)
        axes[2].set_yticks(range(len(bottom_20)))
        axes[2].set_yticklabels(bottom_20['class'], fontsize=8)
        axes[2].set_xlabel('F1-Score')
        axes[2].set_title('Bottom 20 Classes - F1-Score', fontweight='bold')
        axes[2].grid(axis='x', alpha=0.3)
        axes[2].invert_yaxis()
        
        plt.suptitle('Bottom 20 Worst Performing Classes', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
else:
    # Original visualization for fewer classes
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    class_df.plot(x='class', y='precision', kind='bar', ax=axes[0], color='skyblue', legend=False)
    axes[0].set_title('Precision by Class', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Precision')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)
    
    class_df.plot(x='class', y='recall', kind='bar', ax=axes[1], color='lightcoral', legend=False)
    axes[1].set_title('Recall by Class', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Recall')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', alpha=0.3)
    
    class_df.plot(x='class', y='f1', kind='bar', ax=axes[2], color='lightgreen', legend=False)
    axes[2].set_title('F1-Score by Class', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Class')
    axes[2].set_ylabel('F1-Score')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.suptitle('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Generate classification report (only for valid classes)
print(classification_report(y_true, y_pred, labels=valid_classes, zero_division=0))

## Per-Class Performance Analysis

In [ ]:
# Create confusion matrix (only for valid classes from ground truth)
cm = confusion_matrix(y_true, y_pred, labels=valid_classes)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=valid_classes, 
            yticklabels=valid_classes,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix (Valid Classes Only)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.show()

# Calculate and display accuracy for each class
print("\nPer-Class Accuracy:")
print("="*40)
class_accuracy = cm.diagonal() / cm.sum(axis=1)
for i, cls in enumerate(valid_classes):
    print(f"{cls:<20}: {class_accuracy[i]:.4f}")

# Show impact of invalid predictions if any
if metrics['n_invalid_predictions'] > 0:
    print(f"\n⚠️ Note: {metrics['n_invalid_predictions']} invalid predictions are counted as errors")
    print("         but not shown in the confusion matrix above")

In [ ]:
# Create per-class metrics DataFrame (only for valid classes)
class_metrics = []

for i, cls in enumerate(valid_classes):
    cls_mask = y_true == cls
    cls_pred = y_pred[cls_mask]
    
    # Calculate accuracy including invalid predictions as errors
    accuracy = (cls_pred == cls).mean()
    
    class_metrics.append({
        'class': cls,
        'support': cls_mask.sum(),
        'accuracy': accuracy,
        'precision': precision[i],
        'recall': recall[i],
        'f1': f1[i]
    })

class_df = pd.DataFrame(class_metrics)
class_df = class_df.sort_values('f1', ascending=False)

# Display summary statistics
print("="*70)
print("CLASS PERFORMANCE SUMMARY")
print("="*70)
print(f"Total number of valid classes: {len(class_df)}")
print(f"\nF1-Score Statistics:")
print(class_df['f1'].describe().to_string())

if metrics['n_invalid_predictions'] > 0:
    print(f"\n⚠️ Note: {metrics['n_invalid_predictions']} predictions with invalid classes")
    print("         are counted as errors but not shown as separate classes")

# Show top and bottom performers
print("\n" + "="*70)
print("TOP 10 BEST PERFORMING CLASSES")
print("="*70)
top_10 = class_df.nlargest(10, 'f1')
print(top_10[['class', 'f1', 'precision', 'recall', 'support']].to_string(index=False))

print("\n" + "="*70)
print("TOP 10 WORST PERFORMING CLASSES")
print("="*70)
bottom_10 = class_df.nsmallest(10, 'f1')
print(bottom_10[['class', 'f1', 'precision', 'recall', 'support']].to_string(index=False))

# IMPROVED VISUALIZATIONS FOR MANY CLASSES
print(f"\n📊 Creating visualizations for {len(class_df)} classes...")

# Figure 1: Performance Overview - Distribution + Heatmap
fig = plt.figure(figsize=(18, 10))

# Top subplot: Distribution of metrics
gs = fig.add_gridspec(2, 3, height_ratios=[1, 2], hspace=0.3, wspace=0.3)

# Distributions
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(class_df['precision'], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
ax1.axvline(class_df['precision'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {class_df["precision"].mean():.3f}')
ax1.set_xlabel('Precision', fontsize=11)
ax1.set_ylabel('# Classes', fontsize=11)
ax1.set_title('Precision Distribution', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(class_df['recall'], bins=30, color='#e74c3c', edgecolor='black', alpha=0.7)
ax2.axvline(class_df['recall'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {class_df["recall"].mean():.3f}')
ax2.set_xlabel('Recall', fontsize=11)
ax2.set_ylabel('# Classes', fontsize=11)
ax2.set_title('Recall Distribution', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(class_df['f1'], bins=30, color='#2ecc71', edgecolor='black', alpha=0.7)
ax3.axvline(class_df['f1'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {class_df["f1"].mean():.3f}')
ax3.set_xlabel('F1-Score', fontsize=11)
ax3.set_ylabel('# Classes', fontsize=11)
ax3.set_title('F1-Score Distribution', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)

# Bottom subplot: Sorted bar chart showing all metrics
ax4 = fig.add_subplot(gs[1, :])

# Sort by F1 score for better visualization
class_df_sorted = class_df.sort_values('f1', ascending=True)

# Create positions for bars
x_pos = np.arange(len(class_df_sorted))
bar_width = 0.25

# Plot bars
bars1 = ax4.barh(x_pos - bar_width, class_df_sorted['precision'], bar_width, 
                 label='Precision', color='#3498db', alpha=0.8)
bars2 = ax4.barh(x_pos, class_df_sorted['recall'], bar_width,
                 label='Recall', color='#e74c3c', alpha=0.8)
bars3 = ax4.barh(x_pos + bar_width, class_df_sorted['f1'], bar_width,
                 label='F1-Score', color='#2ecc71', alpha=0.8)

# Customize axes
ax4.set_yticks(x_pos[::max(1, len(class_df_sorted)//30)])  # Show only every nth label to avoid overlap
ax4.set_yticklabels(class_df_sorted['class'].iloc[::max(1, len(class_df_sorted)//30)], fontsize=8)
ax4.set_xlabel('Score', fontsize=12, fontweight='bold')
ax4.set_ylabel('Classes (sorted by F1)', fontsize=12, fontweight='bold')
ax4.set_title(f'All Classes Performance ({len(class_df)} classes)', fontsize=14, fontweight='bold')
ax4.legend(loc='lower right', fontsize=11)
ax4.grid(axis='x', alpha=0.3)
ax4.set_xlim([0, 1])

# Add performance zones
ax4.axvspan(0.0, 0.5, alpha=0.1, color='red', label='Poor')
ax4.axvspan(0.5, 0.8, alpha=0.1, color='yellow')
ax4.axvspan(0.8, 1.0, alpha=0.1, color='green')

plt.suptitle('Class Performance Overview', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Figure 2: Top and Bottom Performers (Side by Side)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 performers
top_20 = class_df.nlargest(20, 'f1')
x_pos = np.arange(len(top_20))

axes[0].barh(x_pos - 0.25, top_20['precision'], 0.25, label='Precision', color='#3498db', alpha=0.8)
axes[0].barh(x_pos, top_20['recall'], 0.25, label='Recall', color='#e74c3c', alpha=0.8)
axes[0].barh(x_pos + 0.25, top_20['f1'], 0.25, label='F1-Score', color='#2ecc71', alpha=0.8)
axes[0].set_yticks(x_pos)
axes[0].set_yticklabels(top_20['class'], fontsize=9)
axes[0].set_xlabel('Score', fontsize=11, fontweight='bold')
axes[0].set_title('Top 20 Best Performing Classes', fontsize=13, fontweight='bold', color='green')
axes[0].legend(loc='lower right')
axes[0].grid(axis='x', alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].invert_yaxis()

# Add value labels for F1 scores
for i, (idx, row) in enumerate(top_20.iterrows()):
    axes[0].text(row['f1'] + 0.01, i + 0.25, f"{row['f1']:.3f}", 
                va='center', fontsize=8, color='darkgreen', fontweight='bold')

# Bottom 20 performers
bottom_20 = class_df.nsmallest(20, 'f1').sort_values('f1', ascending=False)
x_pos = np.arange(len(bottom_20))

axes[1].barh(x_pos - 0.25, bottom_20['precision'], 0.25, label='Precision', color='#95a5a6', alpha=0.8)
axes[1].barh(x_pos, bottom_20['recall'], 0.25, label='Recall', color='#e67e22', alpha=0.8)
axes[1].barh(x_pos + 0.25, bottom_20['f1'], 0.25, label='F1-Score', color='#c0392b', alpha=0.8)
axes[1].set_yticks(x_pos)
axes[1].set_yticklabels(bottom_20['class'], fontsize=9)
axes[1].set_xlabel('Score', fontsize=11, fontweight='bold')
axes[1].set_title('Bottom 20 Worst Performing Classes', fontsize=13, fontweight='bold', color='darkred')
axes[1].legend(loc='upper right')
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].invert_yaxis()

# Add value labels for F1 scores
for i, (idx, row) in enumerate(bottom_20.iterrows()):
    axes[1].text(row['f1'] + 0.01, i + 0.25, f"{row['f1']:.3f}", 
                va='center', fontsize=8, color='darkred', fontweight='bold')

plt.suptitle('Best vs Worst Performing Classes', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Figure 3: Performance Heatmap (if many classes)
if len(class_df) > 30:
    # Create a compact heatmap view
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Prepare data for heatmap
    metrics_matrix = class_df.set_index('class')[['precision', 'recall', 'f1']].T
    
    # Sort columns by F1 score
    sorted_cols = class_df.sort_values('f1', ascending=False)['class'].tolist()
    metrics_matrix = metrics_matrix[sorted_cols]
    
    # Create heatmap
    im = ax.imshow(metrics_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    
    # Set ticks
    ax.set_xticks(np.arange(len(sorted_cols)))
    ax.set_yticks(np.arange(3))
    ax.set_yticklabels(['Precision', 'Recall', 'F1-Score'], fontsize=12, fontweight='bold')
    
    # Only show every nth class label to avoid overlap
    step = max(1, len(sorted_cols) // 50)
    ax.set_xticks(np.arange(0, len(sorted_cols), step))
    ax.set_xticklabels([sorted_cols[i] for i in range(0, len(sorted_cols), step)], 
                       rotation=90, ha='right', fontsize=8)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Score', rotation=270, labelpad=15, fontsize=11, fontweight='bold')
    
    # Add title
    ax.set_title(f'Performance Heatmap - All {len(class_df)} Classes (sorted by F1)', 
                fontsize=14, fontweight='bold', pad=20)
    
    # Add grid
    ax.set_xticks(np.arange(len(sorted_cols))-.5, minor=True)
    ax.set_yticks(np.arange(3)-.5, minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

print("\n✅ Visualizations complete!")

## Error Analysis

In [ ]:
# Find misclassifications
errors = df[df[PRED_COLUMN] != df[TRUTH_COLUMN]].copy()

print(f"Total errors: {len(errors):,} ({len(errors)/len(df)*100:.2f}%)")
print(f"Total correct: {len(df) - len(errors):,} ({(len(df) - len(errors))/len(df)*100:.2f}%)")

if len(errors) > 0:
    # Create error pattern
    errors['error_pattern'] = errors[TRUTH_COLUMN].astype(str) + ' → ' + errors[PRED_COLUMN].astype(str)
    
    # Top error patterns
    error_counts = errors['error_pattern'].value_counts().head(10)
    
    print("\nTop 10 Most Common Misclassifications:")
    print("="*50)
    for pattern, count in error_counts.items():
        percentage = count / len(errors) * 100
        print(f"{pattern:<30} : {count:>5} ({percentage:>5.1f}%)")
    
    # Plot error patterns
    plt.figure(figsize=(12, 6))
    error_counts.plot(kind='barh', color='salmon')
    plt.title('Top 10 Error Patterns', fontsize=12, fontweight='bold')
    plt.xlabel('Count')
    plt.ylabel('Error Pattern (Actual → Predicted)')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    
    if SAVE_FIGURES:
        fig_path = OUTPUT_DIR / 'error_patterns.png'
        plt.savefig(fig_path, dpi=FIGURE_DPI, bbox_inches='tight')
        print(f"\n💾 Figure saved to: {fig_path}")
    
    plt.show()

## Export Results

In [ ]:
# Save metrics to CSV
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(BASE_DIR / 'evaluation_metrics.csv', index=False)
print(f"Metrics saved to: {BASE_DIR / 'evaluation_metrics.csv'}")

# Save per-class metrics
class_df.to_csv(BASE_DIR / 'per_class_metrics.csv', index=False)
print(f"Per-class metrics saved to: {BASE_DIR / 'per_class_metrics.csv'}")

# Save confusion matrix (with valid classes only)
cm_df = pd.DataFrame(cm, index=valid_classes, columns=valid_classes)
cm_df.to_csv(BASE_DIR / 'confusion_matrix.csv')
print(f"Confusion matrix saved to: {BASE_DIR / 'confusion_matrix.csv'}")

print("\n✅ All results exported successfully!")